## Diagnóstico inicial (exploratorio — no afecta el resto del pipeline)

In [1]:
import pdfplumber

archivos = {
    'reglamento': '../data/reglamento laliga 2025-2026.pdf',
    'reglas_juego': '../data/Reglas de Juego 2026_27_paginas individuales.pdf',
    'primera_division': '../data/Primera División de España - Wikipedia, la enciclopedia libre.pdf',
    'clasico': '../data/El Clásico - Wikipedia, la enciclopedia libre.pdf',
}

for nombre, ruta in archivos.items():
    with pdfplumber.open(ruta) as pdf:
        n_paginas = len(pdf.pages)
        texto_pagina1 = pdf.pages[0].extract_text() or ""
        texto_pagina_media = pdf.pages[n_paginas // 2].extract_text() or ""
    print(f"--- {nombre} ({n_paginas} páginas) ---")
    print(texto_pagina1[:400])
    print("...")
    print(texto_pagina_media[:300])
    print()

--- reglamento (41 páginas) ---
REGLAMENTO
TEMPORADA 2025/26
...
Artículo 29. Duración del partido
29.1. Todos los partidos dispondrán de cuatro (4) tiempos o partes de diez (10)
minutos cada uno de ellos.
29.2. Cada partido constará de dos intermedios y un descanso. Los intermedios,
que tendrán una duración de 2 minutos, se realizarán entre la primera y
segunda 

--- reglas_juego (276 páginas) ---
Reglas
de Juego
26/27
1
72/6202
ogeuJ
ed
salgeR
...
139

--- primera_division (21 páginas) ---
10/8/26, 5:40 p.m. Primera División de España - Wikipedia, la enciclopedia libre
Primera División de España
La Primera División de España, también conocida como LaLiga EA Sports
por motivos de patrocinio,1 y conocida oficialmente como Campeonato Nacional Primera División de España
de Liga de Primera División, es la máxima categoría masculina del sistema de
ligas de fútbol de España y una de las pr
...
10/8/26, 5:40 p.m. Primera División de España - Wikipedia, la enciclopedia libre
Normativa vigente

In [2]:
import pymupdf  # en vez de: import fitz
doc = pymupdf.open('../data/Reglas de Juego 2026_27_paginas individuales.pdf')  # en vez de: fitz.open(...)
print("Portada:\n", doc[0].get_text()[:300])
print("\nPágina de contenido real (~50):\n", doc[50].get_text()[:500])
doc.close()

Portada:
 1
26/27
Reglas 
de Juego


Página de contenido real (~50):
 51
El punto central del terreno de juego se hallará en el punto medio de la línea 
central, alrededor del cual se trazará una circunferencia con un radio de 9.15 m 
(10 yd).
Se podrán realizar marcaciones fuera del terreno de juego, a 9.15 m (10 yd) del 
arco de esquina y en perpendicular a las líneas de meta y a las líneas de banda.
Todas las líneas deberán tener la misma anchura, como máximo 12 cm (5 in). 
Las líneas de meta tendrán la misma anchura que los postes y el travesaño.
Si un jug


## Funciones de limpieza (cada una definida UNA sola vez, antes de su primer uso)

In [3]:
import re

def limpiar_wikipedia(texto):
    texto_limpio = re.sub(r'', '', texto)
    texto = re.sub(r'\d{1,2}/\d{1,2}/\d{2,4},\s*\d{1,2}:\d{2}\s*[ap]\.m\.\s*.*?Wikipedia, la enciclopedia libre\n?', '', texto)
    texto = re.sub(r'Añadir tema', '', texto)
    texto = re.sub(r'([,\.\)])(\d{1,3})(?=\s|$)', r'\1', texto)  # notas al pie pegadas a puntuación
    return texto

In [4]:
def limpiar_reglas(texto: str) -> str:
    texto = re.sub(r'', '', texto)
    texto = re.sub(r'[\u2000-\u200f\u2028\u2029]', ' ', texto)
    texto = re.sub(r'\x07', '', texto)
    texto = re.sub(r'Reglas de Juego \d{4}/\d{2}[^\n]*\n?', '', texto)  # ya no exige el pipe exacto — borra todo el resto de la línea
    return texto


In [5]:
import pymupdf

def extraer_texto(ruta: str, saltar_primeras: int = 0) -> str:
    doc = pymupdf.open(ruta)
    paginas = [p.get_text() for p in doc[saltar_primeras:]]
    doc.close()
    return '\n'.join(paginas)


In [6]:
def quitar_indice(texto: str) -> str:
    # cualquier línea con una racha larga de puntos es índice, tenga o no sobreviva el número de página
    return re.sub(r'^.*\.{4,}.*$\n?', '', texto, flags=re.MULTILINE)


In [7]:
def normalizar_espacios(texto: str) -> str:
    texto = re.sub(r'[ \t]+', ' ', texto)      # colapsa espacios/tabs repetidos a uno solo
    texto = re.sub(r'\n{3,}', '\n\n', texto)   # colapsa 3+ saltos de línea seguidos a máximo 2 (conserva separación de párrafos)
    return texto.strip()

## Construcción de `textos` — cada documento con su limpiador correcto

In [8]:
textos = {
    'reglamento': normalizar_espacios(quitar_indice(extraer_texto('../data/reglamento laliga 2025-2026.pdf'))),
    'reglas_juego': normalizar_espacios(quitar_indice(limpiar_reglas(extraer_texto('../data/Reglas de Juego 2026_27_paginas individuales.pdf', saltar_primeras=2)))),
    'primera_division': normalizar_espacios(limpiar_wikipedia(extraer_texto('../data/Primera División de España - Wikipedia, la enciclopedia libre.pdf'))),
    'clasico': normalizar_espacios(limpiar_wikipedia(extraer_texto('../data/El Clásico - Wikipedia, la enciclopedia libre.pdf'))),
}

for nombre, texto in textos.items():
    print(f"--- {nombre}: {len(texto)} caracteres ---")
    print(texto[3000:3500])
    print()


--- reglamento: 66490 caracteres ---
ó Barça 
4. RCD Espanyol 
5. Fundación Real Madrid 
6. Fundación Club At. de Madrid 
7. Real Sporting de Gijón 
8. Dépor Genuine 
9. Real Betis Balompié 
10. Real Sociedad Fundazioa 
11. Levante UD EDI 
12. Cádiz CF 
13. Fundació Futbol Base Reus 
14. Real Racing Club 
15. Gimnàstic de Tarragona 
16. RCD Mallorca 
17. Valencia CF 
18. Racing Club de Ferrol 
19. CA Osasuna 
20. Deportivo Alavés 
21. CD Eldense 
22. SD Ponferradina 
23. CD Tenerife EDI 
24. Villarreal CF 
25. Elche CF 
26. Cultura

--- reglas_juego: 280510 caracteres ---
 equipo, pero dado que muchas situaciones son subjetivas y que los 
árbitros son humanos, siempre habrá decisiones erróneas que den lugar al 
debate y la discusión. Hay para quien estos intercambios forman parte del 
encanto del juego, pero, sean las decisiones correctas o incorrectas, el espíritu 
de este deporte requiere que las decisiones arbitrales sean siempre respetadas. 
Los actores que ejercen cierta autoridad

## Splitters

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter_estructurado = RecursiveCharacterTextSplitter(
    chunk_size=800, chunk_overlap=100,
    separators=["\n\nArtículo ", "\n\nRegla ", "\n\n", "\n", ". ", " "],
)

splitter_prosa = RecursiveCharacterTextSplitter(
    chunk_size=800, chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " "],
)


/home/angel/sistema-multiagente-futbol/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Función de chunking con metadata (versión final: ancla encabezados al inicio de línea con `^`, excluye sub-números tipo `3.6` con `(?!\d)`, y usa `.strip()` para no dejar un `\n` sobrante entre título y contenido)

In [10]:
def dividir_con_metadata(texto: str, patron_encabezado: str, splitter, documento: str) -> list[dict]:
    partes = re.split(f'({patron_encabezado})', texto, flags=re.MULTILINE)
    resultado = []
    for i in range(1, len(partes), 2):
        encabezado = partes[i].strip()
        contenido = partes[i + 1].strip() if i + 1 < len(partes) else ''
        texto_completo = f"{encabezado}\n{contenido}"
        sub_chunks = splitter.split_text(texto_completo) if len(texto_completo) > 800 else [texto_completo]
        for chunk in sub_chunks:
            resultado.append({'texto': chunk, 'documento': documento, 'seccion': encabezado})
    return resultado

## Reglas de Juego — diagnóstico de encabezados, luego separación cuerpo principal / anexo de cambios

In [11]:
print(re.findall(r'\n(Regla\s+\d+[^\n]{0,60}|REGLA\s+\d+[^\n]{0,60})', textos['reglas_juego'])[:15])

['Regla 5. El árbitro', 'Regla 3. Los jugadores', 'Regla 3. Los jugadores ', 'Regla 3. Los jugadores', 'Regla 4. El equipamiento de los jugadores', 'Regla 5. El árbitro', 'Regla 6. Los otros miembros del equipo arbitral', 'Regla 8. Inicio y reanudación del juego', 'Regla 10. El resultado de un partido', 'Regla 12. Faltas y conducta incorrecta', 'Regla 14. El penal', 'Regla 15. El saque de banda', 'Regla 16. El saque de meta', 'Regla 3. Los jugadores (p. 64)', 'Regla 3. Los jugadores (p. 65, 66)']


In [12]:
marcador = 'Análisis detallado\nde todos los cambios'
idx_split = textos['reglas_juego'].find(marcador)

cuerpo_reglas = textos['reglas_juego'][:idx_split]
anexo_cambios = textos['reglas_juego'][idx_split:]

# Verificación: ¿queda algún encabezado de regla "real" (sin página) después del corte?
# Si esto sale vacío, confirma que el corte no dejó contenido principal fuera de cuerpo_reglas
print(re.findall(r'\nRegla \d+\.[^\n(]*\n', anexo_cambios)[:5])

[]


## Documentos de Wikipedia (prosa continua, sin estructura de artículo/regla)

In [13]:
chunks_prosa = {
    'primera_division': splitter_prosa.split_text(textos['primera_division']),
    'clasico': splitter_prosa.split_text(textos['clasico']),
}

docs_prosa = [{'texto': c, 'documento': 'primera_division', 'seccion': None} for c in chunks_prosa['primera_division']] + \
             [{'texto': c, 'documento': 'clasico', 'seccion': None} for c in chunks_prosa['clasico']]

print(f"primera_division: {len(chunks_prosa['primera_division'])} chunks")
print(f"clasico: {len(chunks_prosa['clasico'])} chunks")


primera_division: 155 chunks
clasico: 296 chunks


## Chunking final — los 3 documentos estructurados + filtro de longitud mínima (80 caracteres)

In [14]:
docs_estructurados = dividir_con_metadata(textos['reglamento'], r'^Artículo \d+\.(?!\d)[^\n]*', splitter_estructurado, 'reglamento')
docs_reglas_cuerpo = dividir_con_metadata(cuerpo_reglas, r'^Regla \d+\.(?!\d)[^\n(]*', splitter_estructurado, 'reglas_juego')
docs_reglas_cambios = dividir_con_metadata(anexo_cambios, r'^Regla \d+\.[^\n]*\(p\.[^\)]*\)', splitter_estructurado, 'reglas_juego_cambios')

todos_los_chunks = docs_estructurados + docs_reglas_cuerpo + docs_reglas_cambios + docs_prosa
todos_los_chunks = [c for c in todos_los_chunks if len(c['texto']) >= 80]

print(f"Total de chunks: {len(todos_los_chunks)}")
print("Secciones únicas de reglas_juego que contienen un punto seguido de dígito (deberían ser 0 ahora):",
      sum(1 for c in docs_reglas_cuerpo if re.search(r'\d\.\d', c['seccion'])))

Total de chunks: 997
Secciones únicas de reglas_juego que contienen un punto seguido de dígito (deberían ser 0 ahora): 0


## Verificación final: ¿quedó limpio `reglas_juego` (tofu, controles unicode, pie de página)?

In [15]:
print(docs_reglas_cuerpo[3])

{'texto': 'final del medio tiempo (p. ej. sustituciones, lesiones, celebraciones de gol, etc.).\n• Las competiciones deberán decidir quién ayudará al árbitro a controlar el \ntiempo del periodo de exclusión; podría encargarse de ello un delegado, el \ncuarto árbitro o un árbitro asistente neutral; también podría ser un miembro \ndel cuerpo técnico.\n• Una vez concluido el tiempo de la exclusión temporal, el jugador podrá \nregresar por la línea de banda con el permiso del árbitro, cuando el balón \ndeje de estar en juego.', 'documento': 'reglas_juego', 'seccion': 'Regla 5. El árbitro'}


## Modelo de embeddings — prueba con un par de chunks antes de indexar todo

In [16]:
from sentence_transformers import SentenceTransformer
import numpy as np

modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1370.75it/s]


In [17]:
chunk_relevante = next(c for c in todos_los_chunks if 'sustituciones' in c['texto'].lower() and c['documento'] == 'reglas_juego')
chunk_no_relevante = next(c for c in todos_los_chunks if c['documento'] == 'clasico')

pregunta = "¿Cuántos suplentes se pueden usar en un partido?"

vectores = modelo_embeddings.encode([pregunta, chunk_relevante['texto'], chunk_no_relevante['texto']])

def similitud_coseno(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("Similitud con chunk SOBRE sustituciones:   ", similitud_coseno(vectores[0], vectores[1]))
print("Similitud con chunk SOBRE El Clásico:      ", similitud_coseno(vectores[0], vectores[2]))

Similitud con chunk SOBRE sustituciones:    0.43450087
Similitud con chunk SOBRE El Clásico:       0.12867792


## Indexar en ChromaDB — carpeta `chroma_index` (nombre definitivo, sin historial de intentos fallidos), métrica coseno

In [18]:
import chromadb
from chromadb.utils import embedding_functions

embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name='paraphrase-multilingual-MiniLM-L12-v2'
)
client = chromadb.PersistentClient(path='../data/chroma_index')

try:
    client.delete_collection('corpus_laliga')
    print('Colección anterior borrada.')
except Exception:
    print('No había colección previa — primera vez que se crea.')

coleccion = client.create_collection(
    name='corpus_laliga',
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"},
)
coleccion.add(
    ids=[f"chunk_{i}" for i in range(len(todos_los_chunks))],
    documents=[c['texto'] for c in todos_los_chunks],
    metadatas=[{'documento': c['documento'], 'seccion': c['seccion'] or ''} for c in todos_los_chunks],
)
print("Chunks indexados:", coleccion.count())


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1266.29it/s]


No había colección previa — primera vez que se crea.
Chunks indexados: 997


## Pruebas de recuperación

In [19]:
resultados = coleccion.query(query_texts=["¿Cuántos suplentes se pueden usar en un partido?"], n_results=3)
for texto, meta, dist in zip(resultados['documents'][0], resultados['metadatas'][0], resultados['distances'][0]):
    print(f"[{meta['documento']} | {meta['seccion']}] (distancia: {dist:.3f})")
    print(texto[:200], "\n")


[reglas_juego_cambios | Regla 14. El penal (p. 149, 150)] (distancia: 0.381)
221
Terreno de juego (→ field of play [pitch]) 
Superficie delimitada por las líneas de banda y las de meta y, de utilizarse, las 
redes de las porterías.
Tiempo adicional (→ additional time) 
Minutos 

[reglas_juego_cambios | Regla 3. Los jugadores (p. 64)] (distancia: 0.420)
Regla 3. Los jugadores (p. 64)
2. Número de sustituciones
Texto modificado
Otros partidos 
En los partidos de selecciones nacionales absolutas, se podrá designar un 
máximo de quince suplentes, de los 

[reglas_juego | Regla 3. Los jugadores] (distancia: 0.453)
78
Las reglas de la competición podrán estipular otro tipo de restricciones o 
limitaciones relativas al tamaño, número y posición de los eslóganes, mensajes 
e imágenes permitidos. Se recomienda diri 



In [20]:
resultados = coleccion.query(query_texts=["¿Qué pasa si un jugador toca el balón con la mano dentro del área?"], n_results=3)
for texto, meta, dist in zip(resultados['documents'][0], resultados['metadatas'][0], resultados['distances'][0]):
    print(f"[{meta['documento']} | {meta['seccion']}] (distancia: {dist:.3f})")
    print(texto[:200], "\n")


[reglas_juego | Regla 3. Los jugadores] (distancia: 0.183)
se pueda justificar por dicho movimiento. Al colocar su mano o brazo en 
dicha posición, el jugador se arriesga a que el balón golpee esa parte de su 
cuerpo y esto suponga una infracción;
• marque go 

[reglas_juego | Regla 3. Los jugadores] (distancia: 0.192)
contrario, dejará caer el balón ante un jugador del equipo que lo tocó por 
última vez. El balón deberá tocar el suelo en el lugar en el que se 
encontraba cuando se detuvo el juego, a menos que hubie 

[reglas_juego | Regla 3. Los jugadores] (distancia: 0.197)
cuerpo técnico de su propio equipo, se reanudará el juego con un libre 
indirecto desde el punto de la línea de demarcación más cercano al lugar donde 
se cometió la infracción.
Si un jugador contacta 

